# 08 -- Cold-start Random Forest -- strong

Мы решаем cold-start режим -- прогноз нужен сразу после появления ТС, когда длинной истории движения еще нет.

Я сохраняю два production-режима:

- `cold_plus_dev` -- snapshot + расписание + геометрия + `cur_dev_s`;
- `cold_strict` -- тот же online-safe контекст, но без `cur_dev_s`.

Основной CatBoost остается моделью полного контекста после накопления истории.

В этой версии обучение уже серьезное:

- отдельный Optuna study для `cold_plus_dev direct`;
- отдельный study для `cold_plus_dev residual = target_delay_s - cur_dev_s`;
- отдельный study для `cold_strict`;
- 5-fold purged CV;
- отдельный real test;
- learning curves;
- сходимость по числу деревьев;
- стабильность feature importance;
- финальные checkpoints и submissions.

Длинные rolling-окна конкретного ТС, PCA и dataset-only признаки не использую -- cold-start модель должна собираться из данных, которые реально можно получить в online.

In [52]:
from pathlib import Path
import json
import pickle

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OrdinalEncoder

COLD_SEED = 143
COLD_FOLDS = 5
COLD_PURGE_MINUTES = 45

COLD_TRIALS_PER_STUDY = 70
COLD_TUNE_ESTIMATORS = 700
COLD_OOF_ESTIMATORS = 1800
COLD_FINAL_ESTIMATORS = 2500

COLD_TREE_COUNTS = [100, 300, 700, 1200, 1800, 2500]
COLD_SYNTHETIC_WEIGHTS = np.round(np.arange(0, 1.01, 0.05), 2).tolist()
COLD_CLIP_Q = [0.0, 0.0005, 0.001, 0.0025, 0.005, 0.01]

## 1. Данные

Признаки читаю из артефактов `02_feature_engineering.ipynb`.

Здесь нет train-fitted PCA или rank-преобразований -- cold feature space состоит из детерминированных snapshot-, schedule- и физических признаков.

In [53]:
cold_features_dir = Path('../dataset/features')

cold_train = pd.read_pickle(cold_features_dir / 'train_features.pkl')
cold_test = pd.read_pickle(cold_features_dir / 'test_features.pkl')
cold_validate = pd.read_pickle(cold_features_dir / 'validate_features.pkl')

pd.Series({
    'train_rows': len(cold_train),
    'test_rows': len(cold_test),
    'validate_rows': len(cold_validate),
    'train_real': (cold_train.tr_id < 9_000_000).sum(),
    'train_synthetic': (cold_train.tr_id >= 9_000_000).sum(),
})

train_rows         4434
test_rows           353
validate_rows       151
train_real         1141
train_synthetic    3293
dtype: int64

### Online-safe признаки

`cold_strict` может работать сразу после первой нормальной GPS-точки.

`cold_plus_dev` добавляет текущую оценку отклонения от расписания и взаимодействия с ней.

In [54]:
COLD_BASE_NUMERIC = [
    'horizon_s', 'horizon_min',
    'minute_of_day', 'target_minute_of_day',
    'time_sin', 'time_cos', 'target_time_sin', 'target_time_cos',
    'hour', 'target_hour',

    'stop_lon', 'stop_lat',
    'schedule_pos', 'schedule_rows', 'schedule_progress',
    'planned_gap_prev_s', 'planned_gap_next_s',
    'prev_segment_m', 'next_segment_m',
    'planned_speed_prev_kmh', 'planned_speed_next_kmh',
    'stop_visit_count', 'stop_headway_prev_s', 'stop_headway_next_s',
    'prev_stop_lon', 'prev_stop_lat', 'next_stop_lon', 'next_stop_lat',

    'last_speed_kmh',
    'last_heading_deg', 'last_location_valid',
    'last_receive_lag_s', 'last_speed_outlier',
    'packet_age_s', 'event_age_s', 'receive_age_s',

    'last_valid_lon', 'last_valid_lat',
    'last_valid_speed_kmh', 'last_valid_heading_deg',
    'valid_age_s',

    'distance_to_target_m', 'bearing_to_target_deg',
    'heading_error_deg', 'heading_error_abs_deg', 'heading_alignment',
    'last_heading_sin', 'last_heading_cos',
    'bearing_sin', 'bearing_cos',
    'projected_speed_kmh', 'required_speed_kmh',
    'speed_margin_kmh', 'projected_speed_margin_kmh',
    'speed_to_required_ratio',
    'eta_last_speed_s', 'eta_projected_speed_s',
    'eta_margin_last_speed_s', 'eta_margin_projected_s',
    'distance_prev_stop_m', 'distance_next_stop_m',

    'scheduled_stops_to_target',
    'first_future_stop_lon', 'first_future_stop_lat',
    'scheduled_path_m',
    'future_gap_mean_s', 'future_gap_max_s',
    'distance_to_first_future_stop_m',
    'route_proxy_m', 'route_proxy_required_speed_kmh',
    'straight_to_route_ratio',

    'valid_age_gt_30s', 'valid_age_gt_90s',
    'valid_age_gt_5m', 'packet_age_gt_30s',

    'planned_seconds_per_stop',
    'route_m_per_stop', 'scheduled_stops_per_min',
    'planned_vs_straight_distance_m', 'route_straightness',
    'distance_x_valid_age',

    'distance_to_target_m_sqrt',
    'required_speed_kmh_sqrt',
    'valid_age_s_sqrt',
    'packet_age_s_sqrt',
    'route_proxy_m_sqrt',
    'last_valid_speed_kmh_sq',
    'required_speed_kmh_sq',

    'same_grid_4', 'same_grid_3', 'same_grid_2',
]

COLD_CATEGORICAL = [
    'target_grid_4', 'target_grid_3', 'target_grid_2',
    'current_grid_4', 'current_grid_3', 'current_grid_2',
]

COLD_DEV_NUMERIC = [
    'cur_dev_s', 'cur_dev_abs', 'cur_dev_sign',
    'cur_dev_log', 'cur_dev_sqrt', 'cur_dev_sq',
    'cur_dev_per_min',
    'cur_dev_x_horizon',
    'cur_dev_x_required_speed',
    'cur_dev_x_distance_km',
    'cur_dev_per_stop',
    'cur_dev_per_km',
    'cur_dev_plus_eta_margin',
    'cur_dev_x_hour_sin',
    'cur_dev_x_hour_cos',
]

cold_expected = COLD_BASE_NUMERIC + COLD_CATEGORICAL + COLD_DEV_NUMERIC
cold_missing = [cold_col for cold_col in cold_expected if cold_col not in cold_train.columns]

assert not cold_missing, cold_missing

cold_strict_features = COLD_BASE_NUMERIC + COLD_CATEGORICAL
cold_plus_dev_features = cold_strict_features + COLD_DEV_NUMERIC

pd.Series({
    'cold_strict_features': len(cold_strict_features),
    'cold_plus_dev_features': len(cold_plus_dev_features),
    'categorical': len(COLD_CATEGORICAL),
})

cold_strict_features       97
cold_plus_dev_features    112
categorical                 6
dtype: int64

`last_lon_any` и `last_lat_any` намеренно не использую. Online-контракт очищает координаты пакета при `location_valid=False`, поэтому эти два train-признака нельзя воспроизвести строго один в один. Последняя валидная позиция `last_valid_lon/lat` доступна корректно и остается в модели.

## 2. Purged CV

Validation содержит только реальные ТС.

Для каждого `tr_id` временной ряд делю на пять последовательных блоков. Вокруг validation вычищаю из train точки того же ТС на 45 минут -- 30 минут максимальной истории полного пайплайна + 15 минут prediction horizon.

Synthetic используется только в train и получает отдельный вес.

In [55]:
def cold_build_purged_folds(data, n_splits, purge_minutes):
    cold_frame = data[['tr_id', 'T']].copy().reset_index(drop=True)
    cold_frame['T'] = pd.to_datetime(cold_frame['T'])
    cold_frame['is_synthetic'] = cold_frame.tr_id >= 9_000_000
    cold_frame['cv_fold'] = -1

    for cold_tr_id, cold_idx in cold_frame.loc[
        ~cold_frame.is_synthetic
    ].groupby('tr_id').groups.items():
        cold_ordered = (
            cold_frame.loc[cold_idx]
            .sort_values('T')
            .index
            .to_numpy()
            .copy()
        )
        for cold_fold, cold_chunk in enumerate(
            np.array_split(cold_ordered, n_splits)
        ):
            cold_frame.loc[cold_chunk, 'cv_fold'] = cold_fold

    cold_folds = []
    cold_purge = pd.Timedelta(minutes=purge_minutes)

    for cold_fold in range(n_splits):
        cold_valid_mask = (
            (~cold_frame.is_synthetic)
            & (cold_frame.cv_fold == cold_fold)
        )
        cold_train_mask = ~cold_valid_mask

        for cold_tr_id, cold_valid_part in cold_frame.loc[
            cold_valid_mask
        ].groupby('tr_id'):
            cold_left = cold_valid_part['T'].min() - cold_purge
            cold_right = cold_valid_part['T'].max() + cold_purge

            cold_near_valid = (
                (~cold_frame.is_synthetic)
                & (cold_frame.tr_id == cold_tr_id)
                & cold_frame['T'].between(cold_left, cold_right)
            )
            cold_train_mask &= ~cold_near_valid

        cold_folds.append((
            np.flatnonzero(cold_train_mask.to_numpy()),
            np.flatnonzero(cold_valid_mask.to_numpy())
        ))

    return cold_folds

In [56]:
def cold_audit_folds(data, folds, purge_minutes):
    cold_frame = data[['tr_id', 'T']].copy().reset_index(drop=True)
    cold_frame['T'] = pd.to_datetime(cold_frame['T'])

    cold_rows = []

    for cold_fold, (cold_train_idx, cold_valid_idx) in enumerate(folds):
        cold_train_part = cold_frame.iloc[cold_train_idx]
        cold_valid_part = cold_frame.iloc[cold_valid_idx]
        cold_min_gap = np.inf

        for cold_tr_id, cold_valid_group in cold_valid_part.groupby('tr_id'):
            cold_train_times = cold_train_part.loc[
                cold_train_part.tr_id == cold_tr_id,
                'T'
            ]
            if len(cold_train_times) == 0:
                continue

            cold_gap = np.abs(
                cold_train_times.to_numpy()[:, None]
                - cold_valid_group['T'].to_numpy()[None, :]
            ).min()

            cold_min_gap = min(
                cold_min_gap,
                cold_gap / np.timedelta64(1, 'm')
            )

        cold_rows.append({
            'fold': cold_fold,
            'train_rows': len(cold_train_idx),
            'valid_rows': len(cold_valid_idx),
            'valid_tr_ids': cold_valid_part.tr_id.nunique(),
            'valid_synthetic': (
                cold_valid_part.tr_id >= 9_000_000
            ).sum(),
            'min_same_tr_gap_min': cold_min_gap,
        })

    cold_result = pd.DataFrame(cold_rows)

    assert (cold_result.valid_synthetic == 0).all()
    assert (
        cold_result.min_same_tr_gap_min >= purge_minutes
    ).all()

    return cold_result

In [57]:
cold_folds = cold_build_purged_folds(
    cold_train,
    COLD_FOLDS,
    COLD_PURGE_MINUTES
)

cold_fold_audit = cold_audit_folds(
    cold_train,
    cold_folds,
    COLD_PURGE_MINUTES
)

cold_fold_audit

,fold,train_rows,valid_rows,valid_tr_ids,valid_synthetic,min_same_tr_gap_min
0,0,4130,233,13,0,50.0
1,1,4042,231,13,0,50.0
2,2,4082,228,13,0,50.0
3,3,4060,226,13,0,50.0
4,4,4149,223,13,0,50.0


## 3. Preprocessing

Числовые пропуски заполняю медианой внутри каждого fold и добавляю missing indicators.

Пространственные grid-категории кодирую ordinal encoder'ом. Preprocessing validation не видит.

In [58]:
def cold_prepare_frame(data, features):
    cold_frame = data[features].copy()

    for cold_col in COLD_CATEGORICAL:
        if cold_col in cold_frame.columns:
            cold_frame[cold_col] = (
                cold_frame[cold_col]
                .astype('string')
                .fillna('__missing__')
                .astype(str)
            )

    return cold_frame

In [59]:
def cold_make_preprocessor(features):
    cold_cat = [
        cold_col
        for cold_col in COLD_CATEGORICAL
        if cold_col in features
    ]
    cold_num = [
        cold_col
        for cold_col in features
        if cold_col not in cold_cat
    ]

    return ColumnTransformer([
        (
            'numeric',
            SimpleImputer(
                strategy='median',
                add_indicator=True
            ),
            cold_num
        ),
        (
            'categorical',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            cold_cat
        ),
    ])

In [60]:
def cold_sample_weight(data, synthetic_weight):
    return np.where(
        data.tr_id.to_numpy() >= 9_000_000,
        synthetic_weight,
        1.0
    )

In [61]:
def cold_clip_prediction(prediction, y_train, q):
    if q == 0:
        return prediction

    cold_low, cold_high = y_train.quantile(
        [q, 1 - q]
    )

    return np.clip(
        prediction,
        cold_low,
        cold_high
    )

## 4. Search space

В отличие от быстрой версии отдельно тюню все три реально разные задачи.

`absolute_error` оставляю в поиске -- он заметно медленнее `squared_error`, но напрямую оптимизирует L1 в листьях и здесь может быть полезен. Сам лес строит деревья параллельно через `n_jobs=-1`.

In [62]:
def cold_suggest_params(trial):
    cold_bootstrap = trial.suggest_categorical(
        'bootstrap',
        [False, True]
    )

    cold_params = {
        'criterion': trial.suggest_categorical(
            'criterion',
            ['squared_error', 'absolute_error']
        ),
        'max_depth': trial.suggest_categorical(
            'max_depth',
            [None, 12, 16, 20, 24, 32, 40, 48]
        ),
        'min_samples_leaf': trial.suggest_int(
            'min_samples_leaf',
            1,
            10
        ),
        'min_samples_split': trial.suggest_int(
            'min_samples_split',
            2,
            20
        ),
        'max_features': trial.suggest_categorical(
            'max_features',
            [
                'sqrt',
                'log2',
                0.08,
                0.12,
                0.18,
                0.25,
                0.35,
                0.5,
                0.7,
                1.0,
            ]
        ),
        'min_impurity_decrease': trial.suggest_categorical(
            'min_impurity_decrease',
            [0.0, 1e-5, 1e-4, 1e-3, 1e-2]
        ),
        'ccp_alpha': trial.suggest_categorical(
            'ccp_alpha',
            [0.0, 1e-6, 1e-5, 1e-4, 1e-3]
        ),
        'bootstrap': cold_bootstrap,
        'synthetic_weight': trial.suggest_categorical(
            'synthetic_weight',
            COLD_SYNTHETIC_WEIGHTS
        ),
        'clip_q': trial.suggest_categorical(
            'clip_q',
            COLD_CLIP_Q
        ),
    }

    cold_params['max_samples'] = (
        trial.suggest_float(
            'max_samples',
            0.6,
            1.0
        )
        if cold_bootstrap
        else None
    )

    return cold_params

In [63]:
def cold_params_from_trial(trial):
    cold_params = dict(trial.params)
    cold_params.setdefault(
        'max_samples',
        None
    )

    return cold_params

In [64]:
def cold_make_model(
    params,
    n_estimators,
    warm_start=False
):
    return RandomForestRegressor(
        n_estimators=n_estimators,
        criterion=params['criterion'],
        max_depth=params['max_depth'],
        min_samples_leaf=params['min_samples_leaf'],
        min_samples_split=params['min_samples_split'],
        max_features=params['max_features'],
        min_impurity_decrease=params[
            'min_impurity_decrease'
        ],
        ccp_alpha=params['ccp_alpha'],
        bootstrap=params['bootstrap'],
        max_samples=params['max_samples'],
        n_jobs=-1,
        random_state=COLD_SEED,
        warm_start=warm_start
    )

In [65]:
cold_y = cold_train.target_delay_s.astype(float)
cold_cur_dev = cold_train.cur_dev_s.astype(float)

cold_x_plus_dev = cold_prepare_frame(
    cold_train,
    cold_plus_dev_features
)
cold_x_strict = cold_prepare_frame(
    cold_train,
    cold_strict_features
)

In [66]:
def cold_objective(
    trial,
    features,
    x,
    target_mode
):
    cold_params = cold_suggest_params(trial)
    cold_scores = []

    for cold_fold, (
        cold_train_idx,
        cold_valid_idx
    ) in enumerate(cold_folds):
        cold_train_part = cold_train.iloc[
            cold_train_idx
        ]
        cold_weights = cold_sample_weight(
            cold_train_part,
            cold_params['synthetic_weight']
        )
        cold_active = cold_weights > 0

        cold_preprocessor = cold_make_preprocessor(
            features
        )
        cold_train_matrix = cold_preprocessor.fit_transform(
            x.iloc[cold_train_idx].loc[cold_active]
        )
        cold_valid_matrix = cold_preprocessor.transform(
            x.iloc[cold_valid_idx]
        )

        cold_fit_target = cold_y.iloc[
            cold_train_idx
        ].loc[cold_active].copy()

        if target_mode == 'residual':
            cold_fit_target = (
                cold_fit_target
                - cold_cur_dev.iloc[
                    cold_train_idx
                ].loc[cold_active]
            )

        cold_model = cold_make_model(
            cold_params,
            COLD_TUNE_ESTIMATORS
        )
        cold_model.fit(
            cold_train_matrix,
            cold_fit_target,
            sample_weight=cold_weights[cold_active]
        )

        cold_prediction = cold_model.predict(
            cold_valid_matrix
        )

        if target_mode == 'residual':
            cold_prediction += cold_cur_dev.iloc[
                cold_valid_idx
            ].to_numpy()

        cold_prediction = cold_clip_prediction(
            cold_prediction,
            cold_y.iloc[
                cold_train_idx
            ].loc[cold_active],
            cold_params['clip_q']
        )

        cold_scores.append(
            mean_absolute_error(
                cold_y.iloc[cold_valid_idx],
                cold_prediction
            )
        )

        trial.report(
            float(np.mean(cold_scores)),
            step=cold_fold
        )

        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(cold_scores))

## 5. Optuna

Studies хранятся в SQLite и продолжаются после остановки kernel.

Для каждого study создаю отдельный sampler -- один sampler между studies не переиспользую.

In [67]:
cold_artifact_dir = Path(
    '../artifacts/models/cold_start_rf'
)
cold_artifact_dir.mkdir(
    parents=True,
    exist_ok=True
)

cold_storage = (
    f"sqlite:///{cold_artifact_dir / 'optuna_strong.db'}"
)

cold_study_specs = {
    'plus_dev_direct': {
        'features': cold_plus_dev_features,
        'x': cold_x_plus_dev,
        'target_mode': 'direct',
    },
    'plus_dev_residual': {
        'features': cold_plus_dev_features,
        'x': cold_x_plus_dev,
        'target_mode': 'residual',
    },
    'strict_direct': {
        'features': cold_strict_features,
        'x': cold_x_strict,
        'target_mode': 'direct',
    },
}

cold_studies = {}

In [ ]:
for cold_study_index, (
    cold_name,
    cold_spec
) in enumerate(cold_study_specs.items()):
    cold_sampler = optuna.samplers.TPESampler(
        seed=COLD_SEED,
        n_startup_trials=12
    )
    cold_pruner = optuna.pruners.MedianPruner(
        n_startup_trials=12,
        n_warmup_steps=2
    )

    cold_study = optuna.create_study(
        study_name=f'cold_rf_{cold_name}_strong',
        storage=cold_storage,
        load_if_exists=True,
        direction='minimize',
        sampler=cold_sampler,
        pruner=cold_pruner
    )

    cold_finished = sum(
        cold_trial.state in {
            optuna.trial.TrialState.COMPLETE,
            optuna.trial.TrialState.PRUNED,
        }
        for cold_trial in cold_study.trials
    )

    cold_remaining = max(
        0,
        COLD_TRIALS_PER_STUDY - cold_finished
    )

    if cold_remaining:
        cold_study.optimize(
            lambda cold_trial, cold_spec=cold_spec: (
                cold_objective(
                    cold_trial,
                    cold_spec['features'],
                    cold_spec['x'],
                    cold_spec['target_mode']
                )
            ),
            n_trials=cold_remaining,
            n_jobs=1,
            show_progress_bar=False,
            gc_after_trial=True
        )

    cold_studies[cold_name] = cold_study

[I 2026-09-27 17:17:16,779] A new study created in RDB with name: cold_rf_plus_dev_direct_strong
[I 2026-09-27 17:17:38,981] Trial 0 finished with value: 48.80515223915696 and parameters: {'bootstrap': False, 'criterion': 'squared_error', 'max_depth': 40, 'min_samples_leaf': 1, 'min_samples_split': 7, 'max_features': 0.35, 'min_impurity_decrease': 0.001, 'ccp_alpha': 1e-05, 'synthetic_weight': 0.1, 'clip_q': 0.01}. Best is trial 0 with value: 48.80515223915696.
[I 2026-09-27 17:18:24,845] Trial 1 finished with value: 68.0561972838583 and parameters: {'bootstrap': False, 'criterion': 'squared_error', 'max_depth': 24, 'min_samples_leaf': 8, 'min_samples_split': 14, 'max_features': 1.0, 'min_impurity_decrease': 0.001, 'ccp_alpha': 1e-06, 'synthetic_weight': 0.2, 'clip_q': 0.001}. Best is trial 0 with value: 48.80515223915696.
[I 2026-09-27 17:18:35,202] Trial 2 finished with value: 60.77329133023867 and parameters: {'bootstrap': True, 'criterion': 'squared_error', 'max_depth': None, 'min_

In [ ]:
cold_study_table = pd.DataFrame([
    {
        'study': cold_name,
        'trials': len(cold_study.trials),
        'complete': sum(
            cold_trial.state
            == optuna.trial.TrialState.COMPLETE
            for cold_trial in cold_study.trials
        ),
        'best_cv_mae': cold_study.best_value,
    }
    for cold_name, cold_study
    in cold_studies.items()
]).sort_values('best_cv_mae')

cold_study_table

In [ ]:
cold_best_params = {
    cold_name: cold_params_from_trial(
        cold_study.best_trial
    )
    for cold_name, cold_study
    in cold_studies.items()
}

cold_best_params

## 6. OOF на лучших конфигурациях

После Optuna заново строю полноценные 5-fold OOF уже на 1800 деревьях.

Это основная оценка train CV, по ней выбираю direct/residual для `cold_plus_dev`.

In [ ]:
def cold_oof_predict(
    features,
    params,
    target_mode,
    n_estimators
):
    cold_x = cold_prepare_frame(
        cold_train,
        features
    )

    cold_parts = []
    cold_importances = []

    for cold_fold, (
        cold_train_idx,
        cold_valid_idx
    ) in enumerate(cold_folds):
        cold_train_part = cold_train.iloc[
            cold_train_idx
        ]
        cold_weights = cold_sample_weight(
            cold_train_part,
            params['synthetic_weight']
        )
        cold_active = cold_weights > 0

        cold_preprocessor = cold_make_preprocessor(
            features
        )
        cold_train_matrix = cold_preprocessor.fit_transform(
            cold_x.iloc[
                cold_train_idx
            ].loc[cold_active]
        )
        cold_valid_matrix = cold_preprocessor.transform(
            cold_x.iloc[cold_valid_idx]
        )

        cold_fit_target = cold_y.iloc[
            cold_train_idx
        ].loc[cold_active].copy()

        if target_mode == 'residual':
            cold_fit_target = (
                cold_fit_target
                - cold_cur_dev.iloc[
                    cold_train_idx
                ].loc[cold_active]
            )

        cold_model = cold_make_model(
            params,
            n_estimators
        )
        cold_model.fit(
            cold_train_matrix,
            cold_fit_target,
            sample_weight=cold_weights[cold_active]
        )

        cold_prediction = cold_model.predict(
            cold_valid_matrix
        )

        if target_mode == 'residual':
            cold_prediction += cold_cur_dev.iloc[
                cold_valid_idx
            ].to_numpy()

        cold_prediction = cold_clip_prediction(
            cold_prediction,
            cold_y.iloc[
                cold_train_idx
            ].loc[cold_active],
            params['clip_q']
        )

        cold_part = cold_train.iloc[
            cold_valid_idx
        ][
            ['sample_id', 'tr_id', 'T']
        ].copy()
        cold_part['fold'] = cold_fold
        cold_part['target'] = cold_y.iloc[
            cold_valid_idx
        ].to_numpy()
        cold_part['prediction'] = cold_prediction
        cold_part['abs_error'] = np.abs(
            cold_part.target
            - cold_part.prediction
        )
        cold_parts.append(cold_part)

        cold_names = (
            cold_preprocessor
            .get_feature_names_out()
        )
        cold_importances.append(
            pd.Series(
                cold_model.feature_importances_,
                index=cold_names,
                name=cold_fold
            )
        )

    return (
        pd.concat(
            cold_parts,
            ignore_index=True
        ),
        pd.concat(
            cold_importances,
            axis=1
        )
    )

In [ ]:
cold_oof = {}
cold_importance = {}

for cold_name, cold_spec in cold_study_specs.items():
    (
        cold_oof[cold_name],
        cold_importance[cold_name]
    ) = cold_oof_predict(
        cold_spec['features'],
        cold_best_params[cold_name],
        cold_spec['target_mode'],
        COLD_OOF_ESTIMATORS
    )

In [ ]:
cold_oof_metrics = pd.Series({
    cold_name: cold_frame.abs_error.mean()
    for cold_name, cold_frame
    in cold_oof.items()
}).sort_values()

cold_oof_metrics

In [ ]:
cold_plus_candidates = [
    'plus_dev_direct',
    'plus_dev_residual'
]

cold_best_plus_name = (
    cold_oof_metrics
    .loc[cold_plus_candidates]
    .idxmin()
)

cold_best_strict_name = 'strict_direct'

pd.Series({
    'selected_plus_dev': cold_best_plus_name,
    'plus_dev_oof_mae': cold_oof_metrics[
        cold_best_plus_name
    ],
    'strict_oof_mae': cold_oof_metrics[
        cold_best_strict_name
    ],
})

In [ ]:
cold_baseline_median = np.median(
    cold_train.loc[
        cold_train.tr_id < 9_000_000,
        'target_delay_s'
    ]
)

cold_reference_metrics = pd.Series({
    'selected_plus_dev': cold_oof_metrics[
        cold_best_plus_name
    ],
    'cold_strict': cold_oof_metrics[
        cold_best_strict_name
    ],
    'cur_dev_s': mean_absolute_error(
        cold_oof[cold_best_plus_name].target,
        cold_train.set_index('sample_id')
        .loc[
            cold_oof[cold_best_plus_name].sample_id,
            'cur_dev_s'
        ]
        .to_numpy()
    ),
    'global_median': mean_absolute_error(
        cold_oof[cold_best_strict_name].target,
        np.full(
            len(cold_oof[cold_best_strict_name]),
            cold_baseline_median
        )
    ),
}).sort_values()

cold_reference_metrics

In [ ]:
cold_reference_metrics.plot.barh(
    figsize=(8, 4)
)
plt.title('Cold-start -- purged OOF MAE')
plt.xlabel('MAE, sec')
plt.show()

## 7. Стабильность feature importance

In [ ]:
cold_importance_tables = {}

for cold_name in [
    cold_best_plus_name,
    cold_best_strict_name
]:
    cold_table = pd.DataFrame({
        'mean': cold_importance[
            cold_name
        ].mean(axis=1),
        'std': cold_importance[
            cold_name
        ].std(axis=1),
    }).sort_values(
        'mean',
        ascending=False
    )

    cold_importance_tables[
        cold_name
    ] = cold_table

cold_importance_tables[
    cold_best_plus_name
].head(30)

In [ ]:
cold_importance_tables[
    cold_best_plus_name
].head(30).sort_values('mean').plot.barh(
    y='mean',
    xerr='std',
    figsize=(9, 9),
    legend=False
)
plt.title(
    f'Cold-start RF -- importance -- {cold_best_plus_name}'
)
plt.xlabel('mean importance')
plt.show()

## 8. Learning curve

Размер train увеличиваю целыми реальными `tr_id`. Validation остается purged.

Synthetic оставляю с весом, выбранным Optuna.

In [ ]:
def cold_learning_curve(
    features,
    params,
    target_mode,
    fractions
):
    cold_rows = []
    cold_train_idx, cold_valid_idx = cold_folds[-1]

    cold_real_train_ids = (
        cold_train.iloc[cold_train_idx]
        .loc[
            lambda cold_df: (
                cold_df.tr_id < 9_000_000
            ),
            'tr_id'
        ]
        .drop_duplicates()
        .to_numpy()
        .copy()
    )

    cold_rng = np.random.default_rng(
        COLD_SEED
    )
    cold_rng.shuffle(
        cold_real_train_ids
    )

    cold_x = cold_prepare_frame(
        cold_train,
        features
    )

    for cold_fraction in fractions:
        cold_n = max(
            1,
            int(
                np.ceil(
                    len(cold_real_train_ids)
                    * cold_fraction
                )
            )
        )
        cold_selected = set(
            cold_real_train_ids[:cold_n]
        )

        cold_part_idx = np.array([
            cold_idx
            for cold_idx in cold_train_idx
            if (
                cold_train.iloc[cold_idx].tr_id
                >= 9_000_000
                or cold_train.iloc[cold_idx].tr_id
                in cold_selected
            )
        ])

        cold_part = cold_train.iloc[
            cold_part_idx
        ]
        cold_weights = cold_sample_weight(
            cold_part,
            params['synthetic_weight']
        )
        cold_active = cold_weights > 0

        cold_preprocessor = cold_make_preprocessor(
            features
        )
        cold_train_matrix = cold_preprocessor.fit_transform(
            cold_x.iloc[
                cold_part_idx
            ].loc[cold_active]
        )
        cold_valid_matrix = cold_preprocessor.transform(
            cold_x.iloc[cold_valid_idx]
        )

        cold_fit_target = cold_y.iloc[
            cold_part_idx
        ].loc[cold_active].copy()

        if target_mode == 'residual':
            cold_fit_target = (
                cold_fit_target
                - cold_cur_dev.iloc[
                    cold_part_idx
                ].loc[cold_active]
            )

        cold_model = cold_make_model(
            params,
            COLD_OOF_ESTIMATORS
        )
        cold_model.fit(
            cold_train_matrix,
            cold_fit_target,
            sample_weight=cold_weights[cold_active]
        )

        cold_train_prediction = cold_model.predict(
            cold_train_matrix
        )
        cold_valid_prediction = cold_model.predict(
            cold_valid_matrix
        )

        if target_mode == 'residual':
            cold_train_prediction += cold_cur_dev.iloc[
                cold_part_idx
            ].loc[cold_active].to_numpy()
            cold_valid_prediction += cold_cur_dev.iloc[
                cold_valid_idx
            ].to_numpy()

        cold_valid_prediction = cold_clip_prediction(
            cold_valid_prediction,
            cold_y.iloc[
                cold_part_idx
            ].loc[cold_active],
            params['clip_q']
        )

        cold_rows.append({
            'real_train_tr_ids': cold_n,
            'train_rows': int(
                cold_active.sum()
            ),
            'train_mae': mean_absolute_error(
                cold_y.iloc[
                    cold_part_idx
                ].loc[cold_active],
                cold_train_prediction
            ),
            'valid_mae': mean_absolute_error(
                cold_y.iloc[cold_valid_idx],
                cold_valid_prediction
            )
        })

    return pd.DataFrame(
        cold_rows
    )

In [ ]:
cold_plus_learning = cold_learning_curve(
    cold_study_specs[
        cold_best_plus_name
    ]['features'],
    cold_best_params[
        cold_best_plus_name
    ],
    cold_study_specs[
        cold_best_plus_name
    ]['target_mode'],
    [0.25, 0.5, 0.75, 1.0]
)

cold_plus_learning

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    cold_plus_learning.train_rows,
    cold_plus_learning.train_mae,
    marker='o',
    label='train'
)
plt.plot(
    cold_plus_learning.train_rows,
    cold_plus_learning.valid_mae,
    marker='o',
    label='validation'
)
plt.title(
    f'Cold-start RF learning curve -- {cold_best_plus_name}'
)
plt.xlabel('train rows')
plt.ylabel('MAE, sec')
plt.legend()
plt.show()

## 9. Сходимость по числу деревьев

RF не обучается по эпохам, поэтому отдельно смотрю стабилизацию MAE при добавлении деревьев.

Если 1800--2500 деревьев уже дают плато, увеличивать ансамбль дальше не нужно.

In [ ]:
def cold_tree_convergence(
    features,
    params,
    target_mode,
    tree_counts
):
    cold_train_idx, cold_valid_idx = cold_folds[-1]

    cold_train_part = cold_train.iloc[
        cold_train_idx
    ]
    cold_weights = cold_sample_weight(
        cold_train_part,
        params['synthetic_weight']
    )
    cold_active = cold_weights > 0

    cold_x = cold_prepare_frame(
        cold_train,
        features
    )

    cold_preprocessor = cold_make_preprocessor(
        features
    )
    cold_train_matrix = cold_preprocessor.fit_transform(
        cold_x.iloc[
            cold_train_idx
        ].loc[cold_active]
    )
    cold_valid_matrix = cold_preprocessor.transform(
        cold_x.iloc[cold_valid_idx]
    )

    cold_fit_target = cold_y.iloc[
        cold_train_idx
    ].loc[cold_active].copy()

    if target_mode == 'residual':
        cold_fit_target = (
            cold_fit_target
            - cold_cur_dev.iloc[
                cold_train_idx
            ].loc[cold_active]
        )

    cold_model = cold_make_model(
        params,
        tree_counts[0],
        warm_start=True
    )

    cold_rows = []

    for cold_n_estimators in tree_counts:
        cold_model.set_params(
            n_estimators=cold_n_estimators
        )
        cold_model.fit(
            cold_train_matrix,
            cold_fit_target,
            sample_weight=cold_weights[cold_active]
        )

        cold_prediction = cold_model.predict(
            cold_valid_matrix
        )

        if target_mode == 'residual':
            cold_prediction += cold_cur_dev.iloc[
                cold_valid_idx
            ].to_numpy()

        cold_prediction = cold_clip_prediction(
            cold_prediction,
            cold_y.iloc[
                cold_train_idx
            ].loc[cold_active],
            params['clip_q']
        )

        cold_rows.append({
            'n_estimators': cold_n_estimators,
            'valid_mae': mean_absolute_error(
                cold_y.iloc[cold_valid_idx],
                cold_prediction
            )
        })

    return pd.DataFrame(
        cold_rows
    )

In [ ]:
cold_tree_curve = cold_tree_convergence(
    cold_study_specs[
        cold_best_plus_name
    ]['features'],
    cold_best_params[
        cold_best_plus_name
    ],
    cold_study_specs[
        cold_best_plus_name
    ]['target_mode'],
    COLD_TREE_COUNTS
)

cold_tree_curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    cold_tree_curve.n_estimators,
    cold_tree_curve.valid_mae,
    marker='o'
)
plt.title(
    f'Cold-start RF -- tree convergence -- {cold_best_plus_name}'
)
plt.xlabel('n_estimators')
plt.ylabel('purged validation MAE, sec')
plt.show()

## 10. Real test

Test не участвовал ни в Optuna, ни в выборе direct/residual.

Теперь один раз проверяю все три найденные конфигурации на полностью real holdout.

In [ ]:
def cold_fit_predict(
    train_data,
    train_y,
    eval_data,
    features,
    params,
    target_mode,
    n_estimators
):
    cold_train_x = cold_prepare_frame(
        train_data,
        features
    )
    cold_eval_x = cold_prepare_frame(
        eval_data,
        features
    )

    cold_weights = cold_sample_weight(
        train_data,
        params['synthetic_weight']
    )
    cold_active = cold_weights > 0

    cold_preprocessor = cold_make_preprocessor(
        features
    )
    cold_train_matrix = cold_preprocessor.fit_transform(
        cold_train_x.loc[
            cold_active
        ]
    )
    cold_eval_matrix = cold_preprocessor.transform(
        cold_eval_x
    )

    cold_fit_target = train_y.loc[
        cold_active
    ].copy()

    if target_mode == 'residual':
        cold_fit_target = (
            cold_fit_target
            - train_data.loc[
                cold_active,
                'cur_dev_s'
            ].astype(float)
        )

    cold_model = cold_make_model(
        params,
        n_estimators
    )
    cold_model.fit(
        cold_train_matrix,
        cold_fit_target,
        sample_weight=cold_weights[cold_active]
    )

    cold_prediction = cold_model.predict(
        cold_eval_matrix
    )

    if target_mode == 'residual':
        cold_prediction += eval_data[
            'cur_dev_s'
        ].astype(float).to_numpy()

    cold_prediction = cold_clip_prediction(
        cold_prediction,
        train_y.loc[cold_active],
        params['clip_q']
    )

    return (
        cold_model,
        cold_preprocessor,
        cold_prediction
    )

In [ ]:
cold_test_rows = []
cold_test_predictions = {}

for cold_name, cold_spec in cold_study_specs.items():
    (
        cold_model,
        cold_preprocessor,
        cold_prediction
    ) = cold_fit_predict(
        cold_train,
        cold_train.target_delay_s.astype(float),
        cold_test,
        cold_spec['features'],
        cold_best_params[cold_name],
        cold_spec['target_mode'],
        COLD_FINAL_ESTIMATORS
    )

    cold_test_predictions[
        cold_name
    ] = cold_prediction

    cold_test_rows.append({
        'model': cold_name,
        'selected_for_service': (
            cold_name
            in {
                cold_best_plus_name,
                cold_best_strict_name
            }
        ),
        'test_mae': mean_absolute_error(
            cold_test.target_delay_s,
            cold_prediction
        )
    })

cold_test_rows.extend([
    {
        'model': 'cur_dev_s',
        'selected_for_service': False,
        'test_mae': mean_absolute_error(
            cold_test.target_delay_s,
            cold_test.cur_dev_s
        )
    },
    {
        'model': 'global_median',
        'selected_for_service': False,
        'test_mae': mean_absolute_error(
            cold_test.target_delay_s,
            np.full(
                len(cold_test),
                np.median(
                    cold_train.loc[
                        cold_train.tr_id
                        < 9_000_000,
                        'target_delay_s'
                    ]
                )
            )
        )
    },
])

cold_test_metrics = (
    pd.DataFrame(cold_test_rows)
    .sort_values('test_mae')
    .reset_index(drop=True)
)

cold_test_metrics

In [ ]:
cold_test_metrics.plot.barh(
    x='model',
    y='test_mae',
    figsize=(8, 5),
    legend=False
)
plt.title('Cold-start -- real test MAE')
plt.xlabel('MAE, sec')
plt.show()

## 11. Final fit и checkpoints

После unbiased test объединяю train + test.

Для `cold_plus_dev` сохраняю только CV-selected direct/residual постановку. `cold_strict` сохраняю отдельно.

Checkpoint содержит fitted preprocessing, модель, список признаков и mode -- этого достаточно для FastAPI inference.

In [ ]:
cold_all = pd.concat(
    [cold_train, cold_test],
    ignore_index=True
)
cold_all_y = cold_all.target_delay_s.astype(float)

cold_submission_template = pd.read_csv(
    '../dataset/sample_submission.csv',
    sep=';'
)

cold_submission_dir = Path(
    '../submissions/cold_start_rf'
)
cold_submission_dir.mkdir(
    parents=True,
    exist_ok=True
)

cold_checkpoint_dir = (
    cold_artifact_dir
    / 'checkpoints'
)
cold_checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

cold_final_specs = {
    'cold_plus_dev': {
        **cold_study_specs[
            cold_best_plus_name
        ],
        'params': cold_best_params[
            cold_best_plus_name
        ],
        'study_name': cold_best_plus_name,
    },
    'cold_strict': {
        **cold_study_specs[
            cold_best_strict_name
        ],
        'params': cold_best_params[
            cold_best_strict_name
        ],
        'study_name': cold_best_strict_name,
    },
}

cold_final_objects = {}

In [ ]:
for cold_service_name, cold_spec in cold_final_specs.items():
    (
        cold_model,
        cold_preprocessor,
        cold_prediction
    ) = cold_fit_predict(
        cold_all,
        cold_all_y,
        cold_validate,
        cold_spec['features'],
        cold_spec['params'],
        cold_spec['target_mode'],
        COLD_FINAL_ESTIMATORS
    )

    cold_final_weights = cold_sample_weight(
        cold_all,
        cold_spec['params']['synthetic_weight']
    )
    cold_final_active = cold_final_weights > 0
    cold_clip_q = cold_spec['params']['clip_q']

    if cold_clip_q:
        cold_clip_low, cold_clip_high = cold_all_y.loc[
            cold_final_active
        ].quantile([
            cold_clip_q,
            1 - cold_clip_q
        ])
    else:
        cold_clip_low = None
        cold_clip_high = None

    cold_checkpoint = {
        'model': cold_model,
        'preprocessor': cold_preprocessor,
        'features': cold_spec['features'],
        'params': cold_spec['params'],
        'seed': COLD_SEED,
        'mode': cold_service_name,
        'target_mode': cold_spec['target_mode'],
        'study_name': cold_spec['study_name'],
        'n_estimators': COLD_FINAL_ESTIMATORS,
        'clip_low': (
            None
            if cold_clip_low is None
            else float(cold_clip_low)
        ),
        'clip_high': (
            None
            if cold_clip_high is None
            else float(cold_clip_high)
        ),
        'versions': {
            'python_note': 'save training environment separately',
            'numpy': np.__version__,
            'pandas': pd.__version__,
            'sklearn': sklearn.__version__,
        },
    }

    cold_final_objects[
        cold_service_name
    ] = cold_checkpoint

    with open(
        cold_checkpoint_dir
        / f'{cold_service_name}.pkl',
        'wb'
    ) as cold_file:
        pickle.dump(
            cold_checkpoint,
            cold_file
        )

    cold_pred_map = pd.Series(
        cold_prediction,
        index=cold_validate.sample_id
    )

    cold_submission = (
        cold_submission_template.copy()
    )
    cold_submission['prediction'] = (
        cold_submission.sample_id.map(
            cold_pred_map
        )
    )
    cold_submission.to_csv(
        cold_submission_dir
        / f'{cold_service_name}_strong.csv',
        sep=';',
        index=False
    )

sorted(
    cold_checkpoint_dir.glob('*.pkl')
)

In [ ]:
cold_metadata = {
    'seed': COLD_SEED,
    'purge_minutes': COLD_PURGE_MINUTES,
    'final_estimators': COLD_FINAL_ESTIMATORS,
    'cold_plus_dev_study': cold_best_plus_name,
    'cold_plus_dev_oof_mae': float(
        cold_oof_metrics[
            cold_best_plus_name
        ]
    ),
    'cold_strict_oof_mae': float(
        cold_oof_metrics[
            cold_best_strict_name
        ]
    ),
    'service_policy': {
        'cur_dev_available': 'cold_plus_dev',
        'cur_dev_missing': 'cold_strict',
        'enough_history': 'full_catboost',
    },
}

with open(
    cold_artifact_dir
    / 'metadata_strong.json',
    'w',
    encoding='utf-8'
) as cold_file:
    json.dump(
        cold_metadata,
        cold_file,
        ensure_ascii=False,
        indent=2
    )

cold_test_metrics.to_csv(
    cold_artifact_dir
    / 'test_metrics_strong.csv',
    index=False
)

cold_study_table.to_csv(
    cold_artifact_dir
    / 'study_summary_strong.csv',
    index=False
)

for cold_name, cold_frame in cold_oof.items():
    cold_frame.to_csv(
        cold_artifact_dir
        / f'oof_{cold_name}_strong.csv',
        index=False
    )

for cold_name, cold_table in cold_importance_tables.items():
    cold_table.to_csv(
        cold_artifact_dir
        / f'feature_importance_{cold_name}_strong.csv'
    )

for cold_name, cold_study in cold_studies.items():
    cold_study.trials_dataframe().to_csv(
        cold_artifact_dir
        / f'optuna_{cold_name}_trials.csv',
        index=False
    )

cold_artifact_dir

## Итоговая production-логика

```python
if enough_history:
    prediction = full_catboost(...)
elif cur_dev_s is not None:
    prediction = cold_plus_dev(...)
else:
    prediction = cold_strict(...)
```

Сейчас сервис можно поднять уже на двух cold-start checkpoints.

Позже усиленные checkpoints просто заменяются файлами с теми же именами -- FastAPI-контракт и остальная архитектура не меняются.